In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("C:/Users/Administrator/PythonProjects/abfluss_queich")

In [9]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import xarray as xr

from utils.logger import logger

from configs import settings
from jobs.icon.process_icon import clip_to_catchment, extract_precip_timeseries

icon_settings = settings.ingestion.icon
catchment_settings = settings.ingestion.catchment

In [101]:
input_dir = icon_settings.compressed_dir
input_dir = Path(input_dir)

catchment_path = catchment_settings.catchment_path
clip_crs = icon_settings.clip_crs

file_paths = sorted(p for p in input_dir.glob("*.grib2") if "_047" in p.stem)
# file_paths = sorted(p for p in input_dir.glob("*.grib2") if "_048" in p.stem)

catchment = gpd.read_file(catchment_path).to_crs(clip_crs)

In [102]:
# --- Vectorized version NEEDS TO BE CHECKED! ---
def extract_precip_timeseries(
    precip: xr.DataArray,
    ) -> pd.DataFrame:
    
    df = (
        precip
        .mean(dim=[d for d in precip.dims if d != "step"], skipna=True)
        .to_dataframe(name="precip_cum_mean")
        .reset_index()
        .set_index("valid_time")[["precip_cum_mean"]]
    )

    df.index = pd.to_datetime(df.index)
    df.index.name = "timestamp"

    return df

In [ ]:
dfs: list[pd.DataFrame] = []

# --- Open dataset and extract ---
for file in file_paths[:1]:
    try:
        ds = xr.open_dataset(file, engine="cfgrib")

        precip_crop = clip_to_catchment(
            dataset=ds,
            catchment=catchment,
            crs=clip_crs
        )

        df_precip = extract_precip_timeseries(precip=precip_crop)

        dfs.append(df_precip)
    
    
    except Exception:
        logger.exception("Failed to process ICON GRIB file: %s", file)
        continue
    
    
    finally:
        ds.close()
    
        
    if not dfs:
        raise ValueError("No valid precipitation files processed.")
    

                     precip_cum_mean
timestamp                           
2026-05-18 23:00:00         5.357338
2026-05-18 23:15:00         5.357338
2026-05-18 23:30:00         5.357338
2026-05-18 23:45:00         5.357338


In [ ]:
pd.to_datetime(ds.valid_time.values) 

# pd.to_datetime(ds.valid_time.values)

Timestamp('2026-05-19 00:00:00')

In [94]:
df_test = (
    precip_crop
    .mean(dim=["latitude", "longitude"])
    )

df_test

<xarray.DataArray 'tp' ()> Size: 4B
array(5.3573384, dtype=float32)
Coordinates:
    time         datetime64[ns] 8B ...
    step         timedelta64[ns] 8B ...
    surface      float64 8B ...
    valid_time   datetime64[ns] 8B ...
    spatial_ref  int64 8B 0

In [ ]:
172800000000000 * 10**-9 / 60 / 60

48.0

In [55]:
ds

<xarray.Dataset> Size: 4MB
Dimensions:     (latitude: 746, longitude: 1215)
Coordinates:
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
  * latitude    (latitude) float64 6kB 43.18 43.2 43.22 ... 58.04 58.06 58.08
  * longitude   (longitude) float64 10kB -3.94 -3.92 -3.9 ... 20.3 20.32 20.34
    valid_time  datetime64[ns] 8B ...
Data variables:
    tp          (latitude, longitude) float32 4MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             edzw
    GRIB_centreDescription:  Offenbach
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Offenbach
    history:                 2026-06-09T00:08 GRIB to CDM+CF via cfgrib-0.9.1...

In [45]:
for file in file_paths[:1]:
    ds = xr.open_dataset(file, engine="cfgrib")

In [46]:
ds.sizes

Frozen({'step': 4, 'latitude': 746, 'longitude': 1215})